# Step 1: Write your Segmentation Algorithm

Write a function that segments a leaf image, and returns a binary (`dtype='bool'`) image representing the segmentation.  You may add additional parameters besides the image to be segmented if you wish.  Your algorithm must use the random-walker segmentation (``skimage.segmentation.random_walker()``) and you should consider doing some region processing after segmentation to improve the result.

_Hint: The challenge here is to automatically find suitable foreground and background markers for the random walker algorithm.  The noisy images make for an additional challenge, but that's why we're using random walker; becuase of its robustness to noise._

_Hint: When you use `random_walker()` on a color image, make sure to set `multichannel=True`._


In [4]:
import skimage.io as io
import numpy as np
import skimage.segmentation as seg
import skimage.exposure as exposure
import skimage.util as util
import skimage.color as color
import matplotlib.pyplot as plt
import skimage.morphology as morph

def segleaf(I):
    """
    Leaf segmentation using random walker
    :param I: Input image
    
    (Feel free to add as many additional parameters as you
    want/need.)
    
    :return: binary image representing segmentation of a leaf
    """
    
    image = color.rgb2gray(util.img_as_float(I[:, :, 1]))
    markers = np.zeros_like(image, dtype=np.uint)
    # mark all black pixels as background
    markers[image < 0.01] = 2
    # foreground
    markers[image > 0.30] = 1
    # background
    markers[image > 0.75] = 2
    
    labels = seg.random_walker(I, markers, beta=10, multichannel=True)
    binary = labels == 1

    img_processed = morph.remove_small_objects(binary, min_size=20000)
    img_processed = morph.binary_opening(img_processed, selem=morph.disk(4))
    img_processed = morph.remove_small_objects(img_processed, min_size=45000)
    img_processed = morph.remove_small_holes(img_processed, min_size=6000)
    img_processed = morph.binary_closing(img_processed, selem=morph.disk(3))
    
    img_processed, num = morph.label(img_processed, connectivity=2, return_num=True)
    unique, counts = np.unique(img_processed, return_counts=True)
    if len(unique) > 2:
        if counts[1] > counts[2]:
            img_processed = img_processed > 1
        else:
            img_processed = img_processed == 2
    
    return img_processed


# Step 2: Write a Validation driver program.

Write code that segments each image, and computes the DSC for each segmentation.  Print the DSC of each segmentation as you perform it.  At the end, print the average of the DSC over all of the images. 

The general approach should be similar to Assignment 3.  For each input image (in the `images` folder):

* load the image and it's ground truth
* segment the input image 
* Compute the DSC from the segmented image and the ground truth image (this function is given below).
* Print the DSC to the console.

When finished, don't forget to print the average DSC for all images.  If you're getting a mean DSC of greater than 0.8, you're doing pretty good.

_Feel free to define additional helper functions for your program if you think it will help._

In [19]:
import os as os

def dice_coefficient(bwA, bwG):
    '''
    Dice coefficient between two binary images
    :param bwA: a binary (dtype='bool') image
    :param bwG: a binary (dtype='bool') image
    :return: the Dice coefficient between them
    '''
    intersection = np.logical_and(bwA, bwG)

    return 2.0*np.sum(intersection) / (np.sum(bwA) + np.sum(bwG))


# Write your validation driver here.  It should be structured similarly to the driver in Assignment 3.

# Paths for folders -- original and ground truth images
images_path = os.path.join('.', 'noisyimages')
gt_path = os.path.join('.', 'groundtruth')

DSC = []

# Iterate over all files in the original images folder
for root, dirs, files in os.walk(images_path):
    for filename in files:
        # ignore files that are not PNG files.
        if filename[-4:] != '.png':
            continue
            
        # concatenate variable root with filename to get the path to an input file.
        ground_name = os.path.join(gt_path, 'thresh'+filename).replace('\\', '/')
        image_name = os.path.join(images_path, filename).replace('\\', '/')

        img = io.imread(image_name)
        result = segleaf(img)
        
        g_img = io.imread(ground_name)
        g_img = g_img == 255
        
        dsc_value = dice_coefficient(result, g_img)
        
        DSC.append(dsc_value)
        
        print('DSC for ' + filename + ': ' + str(dsc_value))
        print('--------------------------------------------------')

DSC = np.array(DSC)

print('Mean DSC: ' + str(DSC.mean()))
print('Std. Dev of DSC: ' + str(DSC.std()))


C:\Users\Sherry\Anaconda3\lib\site-packages\skimage\morphology\misc.py:207: UserWarning: the min_size argument is deprecated and will be removed in 0.16. Use area_threshold instead.
  "0.16. Use area_threshold instead.")


DSC for image_0001.png: 0.9901144880224378
--------------------------------------------------


DSC for image_0002.png: 0.932776813902557
--------------------------------------------------


DSC for image_0005.png: 0.9897079039890996
--------------------------------------------------


DSC for image_0007.png: 0.983494446428676
--------------------------------------------------


...

DSC for image_0165.png: 0.9846850823208054
--------------------------------------------------


DSC for image_0166.png: 0.9548782129873075
--------------------------------------------------


DSC for image_0171.png: 0.9812022924437019
--------------------------------------------------


DSC for image_0174.png: 0.9764468434639756
--------------------------------------------------


DSC for image_0175.png: 0.9781620911870907
--------------------------------------------------
Mean DSC: 0.9600620229043976
Std. Dev of DSC: 0.05663868921031203


# Step 3:  Display Examples

Choose one input image where your algoirthm performed very well.  Choose another image where the algorithm did not perform well.  Display the two original images with the boundary of the segmentation superimposed on top (just like Step 4 in of assignment 3).  Also display the same two image's ground truth with the segmentation superimposed on top.    Title the images to indicate which is the "good" example, and which is the "bad" example.


In [3]:
import matplotlib.pyplot as plt

good_img = io.imread('noisyimages/image_0100.png')
good_binary = segleaf(good_img)
good_L = morph.label(good_binary, connectivity=2)
boundaries = seg.find_boundaries(good_L, connectivity=2, mode='inner') 
good_img = seg.mark_boundaries(good_img, good_L, color=(1, 0, 0))

bad_img = io.imread('noisyimages/image_0018.png')
bad_binary = segleaf(bad_img)
bad_L = morph.label(bad_binary, connectivity=2)
boundaries = seg.find_boundaries(bad_L, connectivity=2, mode='inner') 
bad_img = seg.mark_boundaries(bad_img, bad_L, color=(1, 0, 0))

good_ground = io.imread('groundtruth/threshimage_0100.png')
good_ground = seg.mark_boundaries(good_ground, good_L, color=(1, 0, 0))

bad_ground = io.imread('groundtruth/threshimage_0018.png')
bad_ground = seg.mark_boundaries(bad_ground, bad_L, color=(1, 0, 0))

plt.imshow(good_img)
plt.title('Good Segmentation Image')
plt.show()
plt.imshow(bad_img)
plt.title('Bad Segmentation Image')
plt.show()

plt.imshow(good_ground)
plt.title('Good Segmentation Ground Image')
plt.show()
plt.imshow(bad_ground)
plt.title('Bad Segmentation Ground Image')
plt.show()

C:\Users\Sherry\Anaconda3\lib\site-packages\skimage\morphology\misc.py:207: UserWarning: the min_size argument is deprecated and will be removed in 0.16. Use area_threshold instead.
  "0.16. Use area_threshold instead.")


# Step 4: Reflection

### Answer the following questions right here in this block.

1. In a few sentences, briefly explain what your segmentation algorithm from Step 1 does and how it works.  

	_Your answer:_  The algorithm firstly segments the image by the random walker method and then it does some region processing to improve the result. The segmentation by random walker is done using markers obtained from grayscaling the green channel of the image. Any pixels that is near black and above a value of 0.75 (lighter pixels) are marked as the background (=2), while any pixels in the midrange (presumably the leaf) are marked as foreground (=1). Some pixels will be left 0 and the random walker method will determine if these pixels will be foreground or background pixels. The region processing includes a few different methods that results in removing small objects, small holes, and large objects and filling the leaf structure.

2. Consider your good result.  What, if anything, about your algoirthm is preventing you from getting a better result with this image?  If you weren't able to get any results, leave this blank, or explain what was preventing you from getting a result.

	_Your answer:_  The small difference between the segmented image and the ground image is that the segmented image appears to have less wavy edges, the top tip of the leaf has been retained and it contains a bit of the stem. Compared to the original image the segmented image appears to have segmented the leaf out well. Therefore it appears that difference in the result is not necessarily that the segmented image is not as good as the ground image but that there are dissimilarities in the segmentation between the segmented image and the ground image.  

3. Consider your bad result.  What is it about your algoirthm caused the poor performance on this image?   If you weren't able to get any results, leave this blank.

	_Your answer:_ My algorithm labels the foreground and background somewhat depending on the 'green' of the pixel. For this particular image, the leaf is touching a cyan coloured strip which would also have a good amount of green within it. Therefore this strip is also considered as the foreground along with the leaf and could not be segmented apart from the leaf by the random walker or the region processing as it was essentially considered a single object.